# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!pip -q install datasets huggingface_hub pandas

In [13]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- One row represents the daily SEO performance of one content item for one client on one report date.
- Table used: fact_content_daily_performance.
- Time window: March 2026 (mid-panel month).
- Prediction target: gsc_clicks.
- Excluded: future information and future observations to avoid data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_impressions
- gsc_avg_position
- ga4_pageviews
- client_has_gsc
- client_has_ga4

## Label
- gsc_clicks

## Context
- report_date
- client_hash_id
- content_hash_id

## Excluded
- Future observations
- Any feature created using future clicks or future pageviews

Reason:
These variables would not be available at prediction time and would introduce data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
import pandas as pd

df = dataset.to_pandas()

print("Rows:", len(df))
print("Columns:", len(df.columns))
print()

print("Date range:")
print(df["report_date"].min(), "to", df["report_date"].max())
print()

print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content:", df["content_hash_id"].nunique())
print()

print("GSC available:")
print(df["gsc_data_available"].value_counts())
print()

print("GA4 available:")
print(df["ga4_data_available"].value_counts())

Rows: 100
Columns: 30

Date range:
2025-01-27 to 2025-01-27

Unique clients: 1
Unique content: 100

GSC available:
gsc_data_available
True    100
Name: count, dtype: int64

GA4 available:
ga4_data_available
False    100
Name: count, dtype: int64


In [15]:
feature_frame = df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "client_has_gsc",
        "client_has_ga4"
    ]
]

feature_frame.head()

,gsc_impressions,gsc_avg_position,ga4_pageviews,client_has_gsc,client_has_ga4
0,30,3.833333,0,True,True
1,5,71.600000,0,True,True
2,1,34.000000,0,True,True
3,6,23.333333,0,True,True
4,5,17.800000,0,True,True


### Feature availability

- gsc_impressions: Available after Search Console processing.
- gsc_avg_position: Available from Search Console reports.
- ga4_pageviews: Available from GA4 daily reporting.
- client_has_gsc: Known before modelling.
- client_has_ga4: Known before modelling.


In [16]:
leak_df = feature_frame.copy()

leak_df["leak_feature"] = df["gsc_clicks"]

print(leak_df.head())

leak_df = leak_df.drop(columns=["leak_feature"])

print("\nLeak feature removed.")
print(leak_df.head())

   gsc_impressions  gsc_avg_position  ga4_pageviews  client_has_gsc  \
0               30          3.833333              0            True   
1                5         71.600000              0            True   
2                1         34.000000              0            True   
3                6         23.333333              0            True   
4                5         17.800000              0            True   

   client_has_ga4  leak_feature  
0            True             0  
1            True             0  
2            True             0  
3            True             0  
4            True             0  

Leak feature removed.
   gsc_impressions  gsc_avg_position  ga4_pageviews  client_has_gsc  \
0               30          3.833333              0            True   
1                5         71.600000              0            True   
2                1         34.000000              0            True   
3                6         23.333333              0           

### Leakage

I created a temporary feature using gsc_clicks to demonstrate data leakage.

Since this feature contains the target information, it would give misleadingly good results.

I removed it before continuing the analysis.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

- This dataset contains historical SEO and analytics information only.
- It cannot explain why rankings changed.
- External factors such as Google algorithm updates, competitor actions, and website changes are not included.
- Results should be interpreted as observational rather than causal.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.